# Tema 5 — Listado II — Ejercicio 1
# Fine-tuning de BERT para clasificación de sentimiento

En este notebook se resuelve el **Ejercicio 1 del Listado II del Tema 5**.

El objetivo es ajustar mediante **fine-tuning** un modelo BERT preentrenado para clasificar comentarios de películas como:

- `neg`: comentario negativo.
- `pos`: comentario positivo.

Usaremos el dataset `rotten_tomatoes` de Hugging Face y el modelo:

```python
bert-base-uncased
```

La estructura del notebook sigue el esquema de la guía proporcionada:

1. Instalar librerías.
2. Cargar el dataset.
3. Obtener las labels.
4. Tokenizar los textos.
5. Cargar `AutoModelForSequenceClassification`.
6. Definir hiperparámetros con `TrainingArguments`.
7. Entrenar con `Trainer`.
8. Evaluar e inferir sobre el conjunto de test.


## 0. Instalación de librerías

En Google Colab puedes ejecutar esta celda.

Si estás trabajando en local con `uv`, puedes instalar las dependencias con:

```bash
uv add transformers datasets evaluate accelerate scikit-learn torch
```

En algunos entornos puede ser necesario reiniciar el kernel después de instalar.


In [34]:
# En Colab, descomenta esta línea si no tienes las librerías instaladas:
# !pip install -q transformers[torch] datasets evaluate accelerate scikit-learn

## 1. Importación de librerías

Importamos las librerías necesarias para:

- Cargar el dataset.
- Tokenizar los textos.
- Cargar BERT para clasificación.
- Configurar el entrenamiento.
- Calcular métricas.
- Realizar inferencia sobre test.


In [35]:
import numpy as np
import torch

from datasets import load_dataset, load_dataset_builder, get_dataset_split_names

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

# Semillas para reproducibilidad aproximada
np.random.seed(42)
torch.manual_seed(42)

print("Versión de torch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())

Versión de torch: 2.12.0+cpu
CUDA disponible: False


## 2. Consultar información del dataset

Primero usamos `load_dataset_builder` para consultar información del dataset sin cargarlo completo todavía.

El dataset que se utilizará es:

```python
rotten_tomatoes
```

Este dataset contiene reseñas cortas de películas con sentimiento positivo o negativo.


In [36]:
ds_builder = load_dataset_builder("rotten_tomatoes")

print("Descripción del dataset:")
print(ds_builder.info.description)

print("\nCaracterísticas del dataset:")
print(ds_builder.info.features)

print("\nSplits disponibles:")
print(get_dataset_split_names("rotten_tomatoes"))

Descripción del dataset:


Características del dataset:
{'text': Value('string'), 'label': ClassLabel(names=['neg', 'pos'])}

Splits disponibles:
['train', 'validation', 'test']


## 3. Cargar el dataset

Ahora sí cargamos el dataset completo.

El dataset ya viene dividido en:

- `train`
- `validation`
- `test`

Por tanto, no necesitamos hacer una partición manual.


In [37]:
dataset = load_dataset("rotten_tomatoes")

print(dataset)

labels = dataset["train"].features["label"].names
NUM_LABELS = len(labels)

print("\nLabels:", labels)
print("Número de labels:", NUM_LABELS)

print("\nEjemplo del conjunto train:")
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
})

Labels: ['neg', 'pos']
Número de labels: 2

Ejemplo del conjunto train:
{'text': 'the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .', 'label': 1}


## 4. Inspección de ejemplos y distribución de clases

Antes de entrenar, conviene revisar algunos ejemplos y comprobar la distribución de clases.

En este dataset:

- `0` corresponde a `neg`.
- `1` corresponde a `pos`.


In [38]:
for i in range(5):
    text = dataset["train"][i]["text"]
    label_id = dataset["train"][i]["label"]
    label_name = labels[label_id]

    print(f"Texto: {text}")
    print(f"Label: {label_id} ({label_name})")
    print("-" * 80)

for split in dataset.keys():
    split_labels = dataset[split]["label"]
    values, counts = np.unique(split_labels, return_counts=True)

    print(f"\nDistribución en {split}:")
    for value, count in zip(values, counts):
        print(f"{value} ({labels[value]}): {count}")

Texto: the rock is destined to be the 21st century's new " conan " and that he's going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .
Label: 1 (pos)
--------------------------------------------------------------------------------
Texto: the gorgeously elaborate continuation of " the lord of the rings " trilogy is so huge that a column of words cannot adequately describe co-writer/director peter jackson's expanded vision of j . r . r . tolkien's middle-earth .
Label: 1 (pos)
--------------------------------------------------------------------------------
Texto: effective but too-tepid biopic
Label: 1 (pos)
--------------------------------------------------------------------------------
Texto: if you sometimes like to go to the movies to have fun , wasabi is a good place to start .
Label: 1 (pos)
--------------------------------------------------------------------------------
Texto: emerges as something rare , an issue movie that's so ho

## 5. Tokenización

BERT no recibe texto directamente. Primero hay que convertir cada frase en tokens numéricos.

Usaremos el tokenizer asociado al modelo:

```python
bert-base-uncased
```

Este modelo es adecuado porque el dataset está en inglés y la versión `uncased` no diferencia mayúsculas y minúsculas.

BERT admite como máximo **512 tokens**, pero las frases de `rotten_tomatoes` suelen ser bastante más cortas. Por eso calcularemos la longitud máxima en train y elegiremos una longitud razonable.


In [39]:
model_id = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_id)

print("Modelo/tokenizer:", model_id)
print("Longitud máxima admitida por el tokenizer:", tokenizer.model_max_length)

Modelo/tokenizer: bert-base-uncased
Longitud máxima admitida por el tokenizer: 512


## 6. Comprobar longitud de los textos

Calculamos cuántos tokens tienen las frases del conjunto de entrenamiento.

Esto ayuda a decidir el valor de `MAX_LENGTH`.

Si usamos una longitud demasiado grande, desperdiciamos memoria.  
Si usamos una longitud demasiado pequeña, truncamos demasiada información.


In [40]:
token_lengths = [
    len(tokenizer(text).input_ids)
    for text in dataset["train"]["text"]
]

print("Longitud mínima:", np.min(token_lengths))
print("Longitud media:", np.mean(token_lengths))
print("Longitud máxima:", np.max(token_lengths))
print("Percentil 90:", np.percentile(token_lengths, 90))
print("Percentil 95:", np.percentile(token_lengths, 95))

# Como el dataset tiene frases cortas, podemos usar directamente la longitud máxima observada en train.
# También podríamos fijar 128 para ir más sobrados.
MAX_LENGTH = max(token_lengths)

print("\nMAX_LENGTH elegido:", MAX_LENGTH)

Longitud mínima: 3
Longitud media: 27.368347010550995
Longitud máxima: 78
Percentil 90: 43.0
Percentil 95: 47.0

MAX_LENGTH elegido: 78


## 7. Función de tokenización

La función `tokenize` recibe un lote de ejemplos y tokeniza el campo `text`.

Usamos:

```python
padding="max_length"
max_length=MAX_LENGTH
truncation=True
```

Aunque hayamos elegido la longitud máxima del train, dejamos `truncation=True` por seguridad. Así evitamos errores si algún texto de validation o test fuera algo más largo.


In [41]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH
    )

## 8. Tokenizar todo el dataset

Aplicamos la función de tokenización a los tres splits del dataset:

- train
- validation
- test

La opción `batched=True` permite tokenizar por lotes y es más eficiente.


In [42]:
encoded_data = dataset.map(tokenize, batched=True)

encoded_data

Map:   0%|          | 0/8530 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1066
    })
})

## 9. Crear subconjuntos pequeños

Para que el entrenamiento sea más rápido, especialmente si se ejecuta en CPU, usamos subconjuntos pequeños.

Puedes aumentar estos valores si tienes GPU o si quieres mejores resultados.

En la guía original se usaban:

```python
1000 ejemplos para train
500 ejemplos para validation
500 ejemplos para test
```


In [43]:
small_train_dataset = encoded_data["train"].shuffle(seed=42).select(range(1000))
small_validation_dataset = encoded_data["validation"].shuffle(seed=42).select(range(500))
small_test_dataset = encoded_data["test"].shuffle(seed=42).select(range(500))

full_train_dataset = encoded_data["train"]
full_validation_dataset = encoded_data["validation"]
full_test_dataset = encoded_data["test"]

print("Small train:", len(small_train_dataset))
print("Small validation:", len(small_validation_dataset))
print("Small test:", len(small_test_dataset))

Small train: 1000
Small validation: 500
Small test: 500


## 10. Comprobar el padding

Comprobamos que todos los textos tokenizados tienen la misma longitud.

Esto es necesario porque los modelos neuronales trabajan con tensores de tamaño fijo dentro de cada batch.


In [44]:
import random

for i in range(10):
    index = random.randint(0, small_train_dataset.num_rows - 1)
    print("text:", index, "len:", len(small_train_dataset[index]["input_ids"]))

text: 654 len: 78
text: 114 len: 78
text: 25 len: 78
text: 759 len: 78
text: 281 len: 78
text: 250 len: 78
text: 228 len: 78
text: 142 len: 78
text: 754 len: 78
text: 104 len: 78


## 11. Preparar formato para PyTorch

El `Trainer` de Hugging Face utiliza tensores.

Por eso indicamos que las columnas relevantes son:

- `input_ids`
- `attention_mask`
- `label`

La columna `text` ya no es necesaria para entrenar.


In [45]:
columns_to_keep = ["input_ids", "attention_mask", "label"]

small_train_dataset.set_format(type="torch", columns=columns_to_keep)
small_validation_dataset.set_format(type="torch", columns=columns_to_keep)
small_test_dataset.set_format(type="torch", columns=columns_to_keep)

print(small_train_dataset[0])

{'label': tensor(0), 'input_ids': tensor([  101,  1012,  1012,  1012,  3248,  2066,  8307, 11867, 13231,  2094,
         6721,  5312,  1997,  1037,  3782,  2600,  9410,  2046,  2054,  2003,
         4728,  1037, 18856, 17322,  1011, 21834,  2094,  2021,  2969,  1011,
         3809,  8645, 10874,  1012,   102,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0]), 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0])}


## 12. Fine-tuning del modelo preentrenado

Cargamos BERT con:

```python
AutoModelForSequenceClassification
```

Esta clase carga el modelo BERT y le añade una cabeza final de clasificación.

Como tenemos dos clases (`neg` y `pos`), indicamos:

```python
num_labels=NUM_LABELS
```

El aviso de que algunos pesos no están inicializados es normal, porque la capa final de clasificación se crea nueva y debe entrenarse.


In [46]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    num_labels=NUM_LABELS,
    id2label={0: "neg", 1: "pos"},
    label2id={"neg": 0, "pos": 1}
)

print("Modelo cargado correctamente.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Modelo cargado correctamente.


## 13. Hiperparámetros

Creamos un objeto de tipo `TrainingArguments`.

Aquí configuramos:

- Carpeta de salida.
- Batch size.
- Número de épocas.
- Estrategia de evaluación.
- Learning rate.
- Desactivación de reportes externos como Weights & Biases.

En versiones recientes de `transformers` el parámetro se llama `eval_strategy`.  
En versiones más antiguas puede llamarse `evaluation_strategy`.

La celda siguiente intenta usar la forma moderna y, si falla, usa la forma antigua.


In [47]:
try:
    args = TrainingArguments(
        output_dir="./outputs",
        report_to="none",
        per_device_train_batch_size=32,
        per_device_eval_batch_size=32,
        num_train_epochs=2,
        learning_rate=2e-5,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=25
    )
except TypeError:
    args = TrainingArguments(
        output_dir="./outputs",
        report_to="none",
        per_device_train_batch_size=32,
        per_device_eval_batch_size=32,
        num_train_epochs=2,
        learning_rate=2e-5,
        weight_decay=0.01,
        evaluation_strategy="epoch",
        save_strategy="no",
        logging_steps=25
    )

args

TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval

## 14. Métricas

Para clasificación de texto suelen usarse:

- Accuracy.
- Precision.
- Recall.
- F1.

Usaremos media `macro`, que calcula la métrica por clase y luego hace la media. Es útil para comparar el rendimiento general entre clases.


In [48]:
def compute_metrics(pred):
    y_true = pred.label_ids
    y_pred = pred.predictions.argmax(-1)

    accuracy = accuracy_score(y_true, y_pred)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

## 15. Crear el `Trainer`

El `Trainer` simplifica el proceso de entrenamiento.

Le pasamos:

- El modelo.
- Los hiperparámetros.
- El conjunto de entrenamiento.
- El conjunto de validación.
- La función de métricas.

En la guía original el `eval_dataset` apuntaba por error al conjunto de entrenamiento pequeño. Aquí usamos correctamente `small_validation_dataset`.


In [49]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=small_train_dataset,
    eval_dataset=small_validation_dataset,
    compute_metrics=compute_metrics
)

print("Trainer creado correctamente.")

Trainer creado correctamente.


## 16. Entrenamiento

Lanzamos el fine-tuning.

Si se ejecuta en CPU, puede tardar varios minutos. Si tarda demasiado, puedes cambiar:

```python
num_train_epochs=1
```

o reducir el tamaño de `small_train_dataset`.


In [50]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.650970,0.503397,0.788000,0.787128,0.802480,0.793474
2,0.462078,0.409367,0.850000,0.849951,0.850506,0.851366


C:\Users\hugo\PycharmProjects\learn-advanced-nlp-deep-learning\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=64, training_loss=0.516283705830574, metrics={'train_runtime': 374.5701, 'train_samples_per_second': 5.339, 'train_steps_per_second': 0.171, 'total_flos': 80166649680000.0, 'train_loss': 0.516283705830574, 'epoch': 2.0})

## 17. Evaluación sobre validation

Después del entrenamiento, evaluamos el modelo sobre el conjunto de validación.

Esto permite comprobar si el modelo ha aprendido antes de pasar a test.


In [51]:
validation_results = trainer.evaluate(small_validation_dataset)

print("Resultados en validation:")
for key, value in validation_results.items():
    print(f"{key}: {value}")

C:\Users\hugo\PycharmProjects\learn-advanced-nlp-deep-learning\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,F1,Precision,Recall
0.462078,0.409367,2,0.850000,0.849951,0.850506,0.851366


Resultados en validation:
eval_loss: 0.4093667268753052
eval_accuracy: 0.85
eval_f1: 0.8499513842484965
eval_precision: 0.8505057294667435
eval_recall: 0.8513660939179541


# 18. Punto 8 — Evaluación e inferencia sobre test

Esta es la parte que el enunciado indica que falta completar.

El objetivo es usar el modelo entrenado para predecir las clases de textos que **no han sido utilizados durante el entrenamiento**.

El proceso es:

1. Usar `trainer.predict(small_test_dataset)`.
2. Obtener los `logits`.
3. Aplicar `softmax` para obtener probabilidades.
4. Aplicar `argmax` para obtener la clase predicha.
5. Comparar con las etiquetas reales.


In [52]:
# Predicción sobre el conjunto de test
test_predictions = trainer.predict(small_test_dataset)

# Logits: puntuaciones sin normalizar que devuelve el modelo
logits = test_predictions.predictions

# Etiquetas reales
y_true = test_predictions.label_ids

# Softmax para convertir logits en probabilidades
probabilities = torch.nn.functional.softmax(
    torch.tensor(logits),
    dim=-1
).numpy()

# Clase predicha: índice de mayor probabilidad
y_pred = np.argmax(probabilities, axis=-1)

print("Forma de logits:", logits.shape)
print("Forma de probabilities:", probabilities.shape)
print("Número de predicciones:", len(y_pred))

C:\Users\hugo\PycharmProjects\learn-advanced-nlp-deep-learning\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Forma de logits: (500, 2)
Forma de probabilities: (500, 2)
Número de predicciones: 500


## 19. Resultados globales sobre test

Ahora calculamos las métricas finales sobre test.

Esta evaluación es importante porque mide el rendimiento sobre ejemplos no usados durante el entrenamiento.


In [53]:
print("Classification report sobre test:")
print(classification_report(
    y_true,
    y_pred,
    target_names=labels,
    zero_division=0
))

print("Matriz de confusión:")
print(confusion_matrix(y_true, y_pred))

test_accuracy = accuracy_score(y_true, y_pred)
test_precision, test_recall, test_f1, _ = precision_recall_fscore_support(
    y_true,
    y_pred,
    average="macro",
    zero_division=0
)

print("\nResumen de métricas en test:")
print("Accuracy:", test_accuracy)
print("Precision macro:", test_precision)
print("Recall macro:", test_recall)
print("F1 macro:", test_f1)

Classification report sobre test:
              precision    recall  f1-score   support

         neg       0.85      0.79      0.82       263
         pos       0.78      0.84      0.81       237

    accuracy                           0.81       500
   macro avg       0.81      0.82      0.81       500
weighted avg       0.82      0.81      0.81       500

Matriz de confusión:
[[208  55]
 [ 38 199]]

Resumen de métricas en test:
Accuracy: 0.814
Precision macro: 0.8144965111068434
Recall macro: 0.8152684859861064
F1 macro: 0.8139397164681357


## 20. Ver predicciones individuales

Mostramos algunas predicciones del conjunto de test con sus probabilidades.

Esto ayuda a entender qué está haciendo el modelo.

Recordatorio:

- `neg`: comentario negativo.
- `pos`: comentario positivo.


In [54]:
# Para poder ver el texto original, usamos el dataset sin formatear
raw_test_dataset = dataset["test"].shuffle(seed=42).select(range(500))

for i in range(10):
    text = raw_test_dataset[i]["text"]
    real_label_id = int(y_true[i])
    pred_label_id = int(y_pred[i])

    real_label = labels[real_label_id]
    pred_label = labels[pred_label_id]

    prob_neg = probabilities[i][0]
    prob_pos = probabilities[i][1]

    print(f"Texto: {text}")
    print(f"Etiqueta real: {real_label}")
    print(f"Predicción: {pred_label}")
    print(f"Probabilidad neg: {prob_neg:.4f}")
    print(f"Probabilidad pos: {prob_pos:.4f}")
    print("-" * 100)

Texto: unpretentious , charming , quirky , original
Etiqueta real: pos
Predicción: pos
Probabilidad neg: 0.2237
Probabilidad pos: 0.7763
----------------------------------------------------------------------------------------------------
Texto: a film really has to be exceptional to justify a three hour running time , and this isn't .
Etiqueta real: neg
Predicción: neg
Probabilidad neg: 0.8627
Probabilidad pos: 0.1373
----------------------------------------------------------------------------------------------------
Texto: working from a surprisingly sensitive script co-written by gianni romoli . . . ozpetek avoids most of the pitfalls you'd expect in such a potentially sudsy set-up .
Etiqueta real: pos
Predicción: neg
Probabilidad neg: 0.7508
Probabilidad pos: 0.2492
----------------------------------------------------------------------------------------------------
Texto: it may not be particularly innovative , but the film's crisp , unaffected style and air of gentle longing make i

## 21. Función auxiliar para predecir textos

Aunque el Ejercicio 1 pide inferencia sobre test, dejamos una función útil para predecir cualquier texto nuevo.

Esta función se utilizará también como base para el Ejercicio 2 del listado.


In [55]:
def predict_sentiment(text):
    # Tokenizamos igual que durante el entrenamiento
    inputs = tokenizer(
        text,
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )

    # Enviamos los tensores al mismo dispositivo que el modelo
    device = model.device
    inputs = {key: value.to(device) for key, value in inputs.items()}

    # Inferencia sin cálculo de gradientes
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    probs = torch.nn.functional.softmax(logits, dim=-1).cpu().numpy()[0]

    pred_id = int(np.argmax(probs))
    pred_label = labels[pred_id]

    return {
        "text": text,
        "pred_id": pred_id,
        "pred_label": pred_label,
        "prob_neg": float(probs[0]),
        "prob_pos": float(probs[1])
    }


example = predict_sentiment("I loved this movie")
example

{'text': 'I loved this movie',
 'pred_id': 1,
 'pred_label': 'pos',
 'prob_neg': 0.23223340511322021,
 'prob_pos': 0.7677665948867798}

## 22. Conclusión

En este ejercicio se ha realizado fine-tuning de `bert-base-uncased` para clasificación binaria de sentimiento sobre el dataset `rotten_tomatoes`.

El modelo BERT ya estaba preentrenado, pero se ha adaptado a la tarea concreta mediante una cabeza de clasificación añadida con `AutoModelForSequenceClassification`.

La parte final del ejercicio consiste en hacer inferencia sobre el conjunto de test. Para ello se han obtenido los logits del modelo, se ha aplicado softmax para convertirlos en probabilidades y se ha utilizado argmax para seleccionar la clase más probable.

Este flujo reproduce el esquema típico de fine-tuning de modelos Transformer para clasificación de textos.
